# Finance Projected Growth Poster Outputs

This notebook creates the boxplot, five-number summary, ranking table, and extra presentation figures for the projected-growth version of the project.


In [ ]:
%pip install pandas matplotlib openpyxl

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/Users/stephenye/stephen/coding/project/datajam')
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORTS_DIR = ROOT / 'reports'
FIGURES_DIR = ROOT / 'figures'
POSTER_ASSETS_DIR = ROOT / 'poster_assets'

JUDGMENT_COLOR = '#C7A663'
PROCESS_COLOR = '#4B5563'
INK = '#1F2933'
BG = '#F6F0E7'

finance = pd.read_csv(PROCESSED_DIR / 'finance_t_test_subset.csv')
model_coefficients = pd.read_csv(PROCESSED_DIR / 'model_coefficients.csv')
model_metrics = pd.read_csv(REPORTS_DIR / 'model_metrics.csv')
finance.head()


In [ ]:
five_number = pd.concat(
    [
        finance.groupby('job_group')['projected_growth_pct'].agg(
            min='min',
            q1=lambda s: s.quantile(0.25),
            median='median',
            q3=lambda s: s.quantile(0.75),
            max='max',
        ),
        pd.DataFrame(
            [
                {
                    'min': finance['projected_growth_pct'].min(),
                    'q1': finance['projected_growth_pct'].quantile(0.25),
                    'median': finance['projected_growth_pct'].median(),
                    'q3': finance['projected_growth_pct'].quantile(0.75),
                    'max': finance['projected_growth_pct'].max(),
                }
            ],
            index=['all_finance_jobs'],
        ),
    ]
).reset_index().rename(columns={'index': 'group'})
five_number


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.6), facecolor=BG)
ax.set_facecolor(BG)
groups = [
    finance.loc[finance['job_group'] == 'judgment-intensive', 'projected_growth_pct'].to_numpy(dtype=float),
    finance.loc[finance['job_group'] == 'process-driven', 'projected_growth_pct'].to_numpy(dtype=float),
]
box = ax.boxplot(groups, labels=['judgment-intensive', 'process-driven'], patch_artist=True, widths=0.55)
for patch, color in zip(box['boxes'], [JUDGMENT_COLOR, PROCESS_COLOR]):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
    patch.set_edgecolor(INK)
for median in box['medians']:
    median.set_color(INK)
    median.set_linewidth(2)
ax.set_title('Projected Growth by Finance Job Group', color=INK, weight='bold')
ax.set_ylabel('Projected Growth % (2024-2034)', color=INK)
ax.grid(axis='y', color='#D8C3A5', alpha=0.65)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 6), facecolor=BG)
ax.set_facecolor(BG)
colors = finance['job_group'].map({'judgment-intensive': JUDGMENT_COLOR, 'process-driven': PROCESS_COLOR})
sizes = finance['annual_openings'].fillna(finance['annual_openings'].median()).to_numpy(dtype=float) / 300
ax.scatter(finance['ai_exposure'], finance['projected_growth_pct'], s=sizes, color=colors, edgecolor=INK, alpha=0.82)
for _, row in finance.iterrows():
    ax.annotate(row['occupation_title'], (row['ai_exposure'], row['projected_growth_pct']), textcoords='offset points', xytext=(5, 5), fontsize=8.4)
ax.set_title('AI Exposure vs Projected Growth in Finance', color=INK, weight='bold')
ax.set_xlabel('AI Exposure', color=INK)
ax.set_ylabel('Projected Growth % (2024-2034)', color=INK)
ax.grid(color='#D8C3A5', alpha=0.65)
plt.show()
